# 11b Classification Embedding UMAP

## Purpose

This notebook is a classification-side embedding exploration notebook.

The main idea here is:
- choose one tokenizer family and one saved model state
- support three model-state options: `pretrained_mlm`, `classification_mlm_init`, and `classification_random_init`
- choose either `mean` or `max` pooling for the sequence embeddings
- embed the prepared labeled glycans from notebook 09
- project the embeddings with UMAP
- color the same embedding space by several broader views:
  - one deterministic subtype view derived from the existing classification labels
  - `N` vs `O`
  - a 3-class broad glycan view
  - broad branching derived from the sequence structure


## Setup note

Same overall pattern as the other notebooks.

- code stays in GitHub
- prepared classification tables and checkpoints stay in Drive
- this notebook reads notebook-09 outputs and existing saved checkpoints
- this notebook does not retrain anything

The subtype task is multi-label, but a scatter plot needs one color per point.
For the subtype-colored UMAP view, the helper script assigns each glycan one
**primary subtype label** by choosing the most-supported label within that
row's existing label set. The original full label set is still kept in the
saved output table.


In [ ]:
# ======================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ======================================================================
import os
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read the prepared classification
# tables and the saved pretrained or classification checkpoints.
drive.mount('/content/drive')

# Keep tqdm in plain-text mode for cleaner notebook logs.
from tqdm.std import tqdm as plain_tqdm
import tqdm.auto as tqdm_auto
tqdm_auto.tqdm = plain_tqdm
try:
    import tqdm.notebook as tqdm_notebook
    tqdm_notebook.tqdm = plain_tqdm
except Exception:
    pass

# UMAP is the only extra package this notebook needs beyond the existing
# project stack used in the other notebooks.
!pip install -q transformers umap-learn

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin main

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


## Choose the tokenizer family, one model state, and one pooling mode

Edit the next cell when you want to switch tokenizer family, swap between the
pretrained MLM checkpoint and the two classifier states, or compare `mean`
versus `max` pooling.


In [ ]:
# ======================================================================
# 1. IMPORT HELPERS AND DEFINE THE ACTIVE RUN
# ======================================================================
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.classification_embedding_umap import (
    SUPPORTED_COLOR_COLUMNS,
    SUPPORTED_MODEL_VARIANTS,
    annotate_classification_umap_metadata,
    build_classification_umap_output_paths,
    build_umap_dataframe,
    compute_umap_projection,
    filter_classification_dataframe_by_split,
    load_combined_classification_splits,
    plot_umap_by_category,
    resolve_embedding_model_dir,
    save_classification_umap_outputs,
    save_json,
    summarize_umap_categories,
)
from src.similarity import build_embedding_lookup_for_dataframe, load_similarity_artifacts

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CLASSIFICATION_PREP_DIR = DRIVE_ROOT / 'results' / 'classification_prep'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
RESULTS_ROOT = DRIVE_ROOT

SUPPORTED_TOKENIZER_FAMILIES = (
    'byte_bpe',
    'glyberta',
    'manual',
    'hybrid_char_bpe',
    'linkage_block',
    'donor_bound',
    'semi_atomic',
)

TOKENIZER_FAMILY = 'manual'
PRETRAIN_EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'
MODEL_VARIANT = 'pretrained_mlm'

# These only matter when MODEL_VARIANT is one of the classification options.
CLASSIFIER_MLM_RUN_LABEL = 'cls_lr2e-5_ep10_bs16_mlm'
CLASSIFIER_RANDOM_RUN_LABEL = 'cls_lr2e-5_ep10_bs16_random_init'
MODEL_SUBDIR = 'best_model'

POOLING_STRATEGY = 'mean'
MAX_LENGTH = 130
BATCH_SIZE = 32
SPLITS_TO_INCLUDE = ('train', 'val', 'test')

# UMAP controls.
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.10
UMAP_METRIC = 'cosine'
RANDOM_SEED = 42

# Color views to summarize for this run.
COLOR_COLUMNS = (
    'primary_subtype_label',
    'n_glycan_vs_other',
    'main_glycan_class',
    'broad_branching',
)

# Label these accessions directly on the binary N-glycan plot.
LABELED_ACCESSIONS = ('G60230HH', 'G27893KR')

# Use a high-contrast palette for the binary N-glycan view.
N_GLYCAN_BINARY_COLORS = {
    'N-glycan': '#0F4C81',
    'Other': '#E76F51',
}

# The subtype view can become crowded. This setting keeps only the top-K most
# frequent subtype labels in the legend and groups the rest under "Other".
SUBTYPE_TOP_K = 18
SUBTYPE_OTHER_LABEL = 'Other subtype'

OUTPUT_RUN_LABEL = 'umap_default'

if TOKENIZER_FAMILY not in SUPPORTED_TOKENIZER_FAMILIES:
    raise ValueError(f'Unsupported TOKENIZER_FAMILY: {TOKENIZER_FAMILY!r}')
if MODEL_VARIANT not in SUPPORTED_MODEL_VARIANTS:
    raise ValueError(f'Unsupported MODEL_VARIANT: {MODEL_VARIANT!r}')
if POOLING_STRATEGY not in {'mean', 'max'}:
    raise ValueError("POOLING_STRATEGY must be either 'mean' or 'max'.")

MODEL_DIR = resolve_embedding_model_dir(
    checkpoints_dir=CHECKPOINTS_DIR,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=PRETRAIN_EXPERIMENT_NAME,
    model_variant=MODEL_VARIANT,
    classifier_mlm_run_label=CLASSIFIER_MLM_RUN_LABEL,
    classifier_random_run_label=CLASSIFIER_RANDOM_RUN_LABEL,
    model_subdir=MODEL_SUBDIR,
)
OUTPUT_PATHS = build_classification_umap_output_paths(
    project_root=RESULTS_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=PRETRAIN_EXPERIMENT_NAME,
    model_variant=MODEL_VARIANT,
    pooling_strategy=POOLING_STRATEGY,
    output_run_label=OUTPUT_RUN_LABEL,
)

TRAIN_CLASSIFICATION_PATH = CLASSIFICATION_PREP_DIR / 'train_classification.csv'
VAL_CLASSIFICATION_PATH = CLASSIFICATION_PREP_DIR / 'val_classification.csv'
TEST_CLASSIFICATION_PATH = CLASSIFICATION_PREP_DIR / 'test_classification.csv'
LABEL_VOCABULARY_PATH = CLASSIFICATION_PREP_DIR / 'label_vocabulary.csv'

run_config = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'pretrain_experiment_name': PRETRAIN_EXPERIMENT_NAME,
    'model_variant': MODEL_VARIANT,
    'classifier_mlm_run_label': CLASSIFIER_MLM_RUN_LABEL,
    'classifier_random_run_label': CLASSIFIER_RANDOM_RUN_LABEL,
    'model_dir': str(MODEL_DIR),
    'pooling_strategy': POOLING_STRATEGY,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'splits_to_include': list(SPLITS_TO_INCLUDE),
    'umap_neighbors': UMAP_NEIGHBORS,
    'umap_min_dist': UMAP_MIN_DIST,
    'umap_metric': UMAP_METRIC,
    'random_seed': RANDOM_SEED,
    'color_columns': list(COLOR_COLUMNS),
    'subtype_top_k': SUBTYPE_TOP_K,
    'subtype_other_label': SUBTYPE_OTHER_LABEL,
    'results_dir': OUTPUT_PATHS['results_dir'],
}
save_json(run_config, OUTPUT_PATHS['run_config_path'])

print(f'Model directory: {MODEL_DIR}')
print(f'Results directory: {OUTPUT_PATHS["results_dir"]}')


## Load the prepared classification tables and derive broader label views

This is the handoff from notebook 09. The helper module combines the saved
train, validation, and test classification tables, then derives a few broader
views for UMAP coloring.

Those broad views are:
- `primary_subtype_label`: one deterministic subtype label chosen for plotting
- `n_glycan_vs_other`: a binary `N-glycan` versus `Other` view
- `main_glycan_class`: a broader glycan-class view
- `broad_branching`: `Unbranched`, `Single branch`, `Two branches`, or `Highly branched`

This run also labels `G60230HH` and `G27893KR` directly on the binary
`N-glycan` versus `Other` plot.


In [ ]:
# ======================================================================
# 2. LOAD NOTEBOOK-09 OUTPUTS AND BUILD THE UMAP METADATA TABLE
# ======================================================================
classification_df = load_combined_classification_splits(
    train_csv_path=TRAIN_CLASSIFICATION_PATH,
    val_csv_path=VAL_CLASSIFICATION_PATH,
    test_csv_path=TEST_CLASSIFICATION_PATH,
)
classification_df = filter_classification_dataframe_by_split(
    classification_df=classification_df,
    splits_to_include=SPLITS_TO_INCLUDE,
)
label_vocabulary_df = pd.read_csv(LABEL_VOCABULARY_PATH)
annotated_df = annotate_classification_umap_metadata(
    classification_df=classification_df,
    label_vocabulary_df=label_vocabulary_df,
)
category_summary_df = summarize_umap_categories(annotated_df, category_columns=COLOR_COLUMNS)

print(f'Rows included in this run: {len(annotated_df)}')
print('Split counts')
display(annotated_df['split'].value_counts().rename_axis('split').reset_index(name='count'))
print('Broad-category counts')
display(category_summary_df)


## Load the model and build the sequence embeddings

This notebook reuses the similarity-side embedding path so the exact same
checkpoint can be inspected with either `mean` or `max` pooling.


In [ ]:
# ======================================================================
# 3. LOAD THE CHECKPOINT AND EMBED THE GLYCANS
# ======================================================================
tokenizer, model, runtime_device = load_similarity_artifacts(str(MODEL_DIR))
embedding_bundle = build_embedding_lookup_for_dataframe(
    sequence_df=annotated_df,
    tokenizer=tokenizer,
    model=model,
    accession_col='glycan_id',
    sequence_col='sequence',
    device=runtime_device,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    pooling_strategy=POOLING_STRATEGY,
)

print(f'Runtime device: {runtime_device}')
print(f'Unique sequences embedded: {len(embedding_bundle["unique_sequences"])}')
print(f'Pooling strategy: {embedding_bundle["pooling_strategy"]}')


## Run UMAP and attach the coordinates

This step projects the pooled sequence embeddings into two dimensions while
keeping the derived category columns beside each row for later plotting and
manual review.


In [ ]:
# ======================================================================
# 4. COMPUTE THE UMAP PROJECTION
# ======================================================================
umap_coordinates = compute_umap_projection(
    embeddings=embedding_bundle['normalized_embeddings'],
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_SEED,
)
umap_df = build_umap_dataframe(
    annotated_df=embedding_bundle['sequence_df'],
    umap_coordinates=umap_coordinates,
)

print('UMAP preview')
display(
    umap_df[
        [
            'glycan_id',
            'sequence',
            'split',
            'primary_subtype_label',
            'n_o_category',
            'main_glycan_class',
            'broad_branching',
            'umap_1',
            'umap_2',
        ]
    ].head(10)
)


## Save one UMAP plot per requested color view

The subtype plot can become visually busy, so only the top-supported subtype
labels keep their own legend entries by default. The broader category views
are saved in full.


In [ ]:
# ======================================================================
# 5. SAVE THE UMAP PLOTS
# ======================================================================
plot_paths = {
    'primary_subtype_label': plot_umap_by_category(
        umap_df=umap_df,
        category_column='primary_subtype_label',
        output_path=OUTPUT_PATHS['primary_subtype_plot_path'],
        title='UMAP colored by primary subtype label',
        top_k_categories=SUBTYPE_TOP_K,
        other_label=SUBTYPE_OTHER_LABEL,
    ),
    'n_glycan_vs_other': plot_umap_by_category(
        umap_df=umap_df,
        category_column='n_glycan_vs_other',
        output_path=OUTPUT_PATHS['n_glycan_vs_other_plot_path'],
        title='UMAP colored by N-glycan vs Other',
        category_colors=N_GLYCAN_BINARY_COLORS,
        label_accessions=LABELED_ACCESSIONS,
        point_size=20,
        alpha=0.88,
    ),
    'main_glycan_class': plot_umap_by_category(
        umap_df=umap_df,
        category_column='main_glycan_class',
        output_path=OUTPUT_PATHS['main_class_plot_path'],
        title='UMAP colored by broad 3-class glycan category',
    ),
    'broad_branching': plot_umap_by_category(
        umap_df=umap_df,
        category_column='broad_branching',
        output_path=OUTPUT_PATHS['branching_plot_path'],
        title='UMAP colored by broad branching category',
    ),
}

for plot_name, plot_path in plot_paths.items():
    print(f'{plot_name}: {plot_path}')


## Save the annotated table and show the saved plots

The saved CSV keeps the original label set, the broad derived views, and the
UMAP coordinates together so the run is easy to review later without rerunning
embedding extraction.


In [ ]:
# ======================================================================
# 6. SAVE TABLES AND DISPLAY THE PLOTS
# ======================================================================
saved_paths = save_classification_umap_outputs(
    umap_df=umap_df,
    category_summary_df=category_summary_df,
    output_paths=OUTPUT_PATHS,
)

print('Saved output paths')
for key, value in saved_paths.items():
    print(f'- {key}: {value}')

for image_path in (
    OUTPUT_PATHS['primary_subtype_plot_path'],
    OUTPUT_PATHS['n_glycan_vs_other_plot_path'],
    OUTPUT_PATHS['main_class_plot_path'],
    OUTPUT_PATHS['branching_plot_path'],
):
    display(Image(filename=image_path))


## Final note

At this point I should have:
- one saved annotated glycan table with UMAP coordinates
- one category-count summary table
- four saved UMAP plots
- a reusable parameter block for swapping tokenizer family, model state, pooling mode, and split selection

If I want to compare multiple checkpoints side by side, I can rerun this same
notebook with a different `MODEL_VARIANT`, `POOLING_STRATEGY`, or tokenizer
family while keeping the rest of the workflow unchanged.
